In [24]:
# ============================================================
# PHASE 4 — CELL 1
# Imports + Environment
# ============================================================

from pathlib import Path
import os
import json
import csv
import traceback
import gc

import cv2
import numpy as np
import pandas as pd

import mediapipe as mp


print("=" * 70)
print("PHASE 4 — ENVIRONMENT")
print("=" * 70)

print("Python environment ready")
print("NumPy :", np.__version__)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)

print("=" * 70)

PHASE 4 — ENVIRONMENT
Python environment ready
NumPy : 1.26.4
OpenCV: 4.11.0
MediaPipe: 0.10.14


In [25]:
# ============================================================
# PHASE 4 — CELL 2
# Project Paths
# ============================================================

PROJECT_ROOT = Path(
    r"C:\LipReadingSSL"
).resolve()

PHASE4_OUTPUT_ROOT = (
    PROJECT_ROOT / "output"
).resolve()


SUPPORTED_VIDEO_EXTENSIONS = {
    ".webm",
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
}


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
}


print("=" * 70)
print("PHASE 4 — PATH CONFIGURATION")
print("=" * 70)

print("Project root:")
print(PROJECT_ROOT)

print()
print("Phase 4 output root:")
print(PHASE4_OUTPUT_ROOT)

if not PHASE4_OUTPUT_ROOT.exists():

    raise FileNotFoundError(
        f"Output root not found:\n"
        f"{PHASE4_OUTPUT_ROOT}"
    )

print()
print("✅ Path configuration ready")

print("=" * 70)

PHASE 4 — PATH CONFIGURATION
Project root:
C:\LipReadingSSL

Phase 4 output root:
C:\LipReadingSSL\output

✅ Path configuration ready


In [26]:
# ============================================================
# PHASE 4 — CELL 3
# Phase 3 Output Paths
# ============================================================


def get_phase4_paths(
    video_output_dir
):

    video_output_dir = (
        Path(video_output_dir)
        .resolve()
    )

    return {

        # ----------------------------------------------------
        # Phase 3 INPUT
        # ----------------------------------------------------

        "aligned_face":
            video_output_dir /
            "aligned_face",

        "landmarks":
            video_output_dir /
            "landmarks",

        "alignment_metadata":
            video_output_dir /
            "alignment_metadata.csv",

        # ----------------------------------------------------
        # Phase 4 OUTPUT
        # ----------------------------------------------------

        "mouth_crop":
            video_output_dir /
            "mouth_crop",

        "mouth_metadata":
            video_output_dir /
            "mouth_metadata.csv",

        "preview":
            video_output_dir /
            "phase4_preview",

    }


print("=" * 70)
print("Cell 3 — Phase 4 Path System Ready")
print("=" * 70)

Cell 3 — Phase 4 Path System Ready


In [27]:
# ============================================================
# PHASE 4 — CELL 4
# MediaPipe Face Mesh
# ============================================================

mp_face_mesh = mp.solutions.face_mesh


FACE_MESH_STATIC_IMAGE_MODE = True

FACE_MESH_MAX_NUM_FACES = 1

FACE_MESH_REFINE_LANDMARKS = True

FACE_MESH_MIN_DETECTION_CONFIDENCE = 0.5

FACE_MESH_MIN_TRACKING_CONFIDENCE = 0.5


face_mesh = mp_face_mesh.FaceMesh(

    static_image_mode=
        FACE_MESH_STATIC_IMAGE_MODE,

    max_num_faces=
        FACE_MESH_MAX_NUM_FACES,

    refine_landmarks=
        FACE_MESH_REFINE_LANDMARKS,

    min_detection_confidence=
        FACE_MESH_MIN_DETECTION_CONFIDENCE,

    min_tracking_confidence=
        FACE_MESH_MIN_TRACKING_CONFIDENCE,
)


print("=" * 70)
print("Cell 4 — MediaPipe Face Mesh Ready")
print("=" * 70)

Cell 4 — MediaPipe Face Mesh Ready


In [28]:
# ============================================================
# PHASE 4 — CELL 5
# Mouth Landmark Configuration
# ============================================================


MOUTH_LANDMARKS = [

    61, 146, 91, 181, 84, 17,
    314, 405, 321, 375, 291, 308,
    324, 318, 402, 317, 14, 87,
    178, 88, 95, 185, 40,
    39, 37, 0, 267, 269, 270,
    409, 415, 310, 311, 312, 13,
    82, 81, 42, 183, 78

]


MOUTH_PADDING = 20

MOUTH_SIZE = (
    96,
    96
)


print("=" * 70)
print("Cell 5 — Mouth Landmark Configuration")
print("=" * 70)

print(
    "Mouth landmarks:",
    len(MOUTH_LANDMARKS)
)

print(
    "Padding:",
    MOUTH_PADDING
)

print(
    "Output size:",
    MOUTH_SIZE
)

print("=" * 70)

Cell 5 — Mouth Landmark Configuration
Mouth landmarks: 40
Padding: 20
Output size: (96, 96)


In [29]:
# ============================================================
# PHASE 4 — CELL 6
# Output Structure
# ============================================================


def create_phase4_output_structure(
    video_output_dir
):

    paths = get_phase4_paths(
        video_output_dir
    )

    paths["mouth_crop"].mkdir(
        parents=True,
        exist_ok=True
    )

    paths["preview"].mkdir(
        parents=True,
        exist_ok=True
    )

    return paths


print("=" * 70)
print("Cell 6 — Output Structure Ready")
print("=" * 70)

Cell 6 — Output Structure Ready


In [30]:
# ============================================================
# PHASE 4 — CELL 7
# Discover Aligned Frames
# ============================================================


def extract_frame_index(
    path
):

    stem = Path(path).stem

    digits = "".join(
        ch
        for ch in stem
        if ch.isdigit()
    )

    if not digits:
        return None

    return int(digits)


def discover_aligned_frames(
    aligned_dir
):

    aligned_dir = Path(
        aligned_dir
    )

    frame_map = {}

    if not aligned_dir.exists():
        return frame_map

    for path in aligned_dir.iterdir():

        if not path.is_file():
            continue

        if (
            path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        frame_index = (
            extract_frame_index(path)
        )

        if frame_index is None:
            continue

        frame_map[
            frame_index
        ] = path

    return frame_map


print("=" * 70)
print("Cell 7 — Aligned Frame Discovery Ready")
print("=" * 70)

Cell 7 — Aligned Frame Discovery Ready


In [31]:
# ============================================================
# PHASE 4 — CELL 8
# Load Alignment Metadata
# ============================================================


REQUIRED_ALIGNMENT_COLUMNS = {

    "video_id",
    "frame_index",

}


def load_alignment_metadata(
    metadata_path
):

    metadata_path = Path(
        metadata_path
    )

    if not metadata_path.exists():

        raise FileNotFoundError(
            f"Alignment metadata not found:\n"
            f"{metadata_path}"
        )

    df = pd.read_csv(
        metadata_path
    )

    missing = (
        REQUIRED_ALIGNMENT_COLUMNS
        - set(df.columns)
    )

    if missing:

        raise ValueError(
            "Alignment metadata missing "
            f"columns: {sorted(missing)}"
        )

    df["frame_index"] = pd.to_numeric(
        df["frame_index"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["frame_index"]
    )

    df["frame_index"] = (
        df["frame_index"]
        .astype(int)
    )

    df = df.drop_duplicates(
        subset=["frame_index"],
        keep="first"
    )

    df = df.sort_values(
        "frame_index"
    ).reset_index(
        drop=True
    )

    return df


print("=" * 70)
print("Cell 8 — Alignment Metadata Loader Ready")
print("=" * 70)

Cell 8 — Alignment Metadata Loader Ready


In [32]:
# ============================================================
# PHASE 4 — CELL 9
# MediaPipe Mouth Crop
# ============================================================


def crop_mouth_from_aligned_face(
    image
):

    if image is None:
        return None, "invalid_image"

    if image.size == 0:
        return None, "empty_image"

    try:

        rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

    except Exception:

        return None, "rgb_conversion_error"


    try:

        result = face_mesh.process(
            rgb
        )

    except Exception:

        return None, "mediapipe_error"


    if (
        not result.multi_face_landmarks
    ):

        return None, "no_landmarks"


    landmarks = (
        result
        .multi_face_landmarks[0]
        .landmark
    )


    h, w = image.shape[:2]

    points = []


    for index in MOUTH_LANDMARKS:

        if index >= len(landmarks):

            return None, "invalid_landmark_index"

        x = int(
            landmarks[index].x * w
        )

        y = int(
            landmarks[index].y * h
        )

        points.append(
            [x, y]
        )


    points = np.asarray(
        points,
        dtype=np.int32
    )


    x_min = max(
        0,
        int(points[:, 0].min())
        - MOUTH_PADDING
    )

    y_min = max(
        0,
        int(points[:, 1].min())
        - MOUTH_PADDING
    )

    x_max = min(
        w,
        int(points[:, 0].max())
        + MOUTH_PADDING
    )

    y_max = min(
        h,
        int(points[:, 1].max())
        + MOUTH_PADDING
    )


    if (
        x_max <= x_min
        or
        y_max <= y_min
    ):

        return None, "invalid_crop"


    mouth = image[
        y_min:y_max,
        x_min:x_max
    ]


    if mouth.size == 0:

        return None, "empty_crop"


    try:

        mouth = cv2.resize(
            mouth,
            MOUTH_SIZE,
            interpolation=cv2.INTER_AREA
        )

    except Exception:

        return None, "resize_error"


    return mouth, "OK"


print("=" * 70)
print("Cell 9 — MediaPipe Mouth Crop Ready")
print("=" * 70)

Cell 9 — MediaPipe Mouth Crop Ready


In [33]:
# ============================================================
# PHASE 4 — CELL 10
# Process One Video
# ============================================================


MOUTH_METADATA_COLUMNS = [

    "video_id",
    "frame_index",
    "aligned_face",
    "mouth_crop",
    "status",
    "reason",

]


def process_video(
    video_output_dir
):

    video_output_dir = Path(
        video_output_dir
    ).resolve()

    video_id = (
        video_output_dir.name
    )


    paths = (
        create_phase4_output_structure(
            video_output_dir
        )
    )


    aligned_dir = (
        paths["aligned_face"]
    )

    metadata_path = (
        paths["alignment_metadata"]
    )

    mouth_dir = (
        paths["mouth_crop"]
    )

    mouth_metadata_path = (
        paths["mouth_metadata"]
    )


    print()
    print("=" * 70)
    print(
        f"PHASE 4 PROCESSING : {video_id}"
    )
    print("=" * 70)

    print(
        "Aligned face :",
        aligned_dir
    )

    print(
        "Metadata     :",
        metadata_path
    )

    print(
        "Mouth crop   :",
        mouth_dir
    )


    if not aligned_dir.exists():

        raise FileNotFoundError(
            f"Aligned face directory not found:\n"
            f"{aligned_dir}"
        )


    metadata_df = (
        load_alignment_metadata(
            metadata_path
        )
    )


    aligned_map = (
        discover_aligned_frames(
            aligned_dir
        )
    )


    print()
    print(
        "Metadata frames :",
        len(metadata_df)
    )

    print(
        "Aligned frames  :",
        len(aligned_map)
    )


    rows = []


    stats = {

        "video_id":
            video_id,

        "input_frames":
            len(metadata_df),

        "mouth_saved":
            0,

        "skipped":
            0,

        "no_landmarks":
            0,

        "invalid_crop":
            0,

        "errors":
            0,

    }


    # --------------------------------------------------------
    # Process by metadata frame_index
    # --------------------------------------------------------

    for _, metadata_row in metadata_df.iterrows():

        frame_index = int(
            metadata_row[
                "frame_index"
            ]
        )


        aligned_path = (
            aligned_map.get(
                frame_index
            )
        )


        # ----------------------------------------------------
        # Missing aligned frame
        # ----------------------------------------------------

        if aligned_path is None:

            stats["skipped"] += 1

            rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "aligned_face":
                    "",

                "mouth_crop":
                    "",

                "status":
                    "SKIPPED",

                "reason":
                    "missing_aligned_face",

            })

            continue


        # ----------------------------------------------------
        # Already processed
        # ----------------------------------------------------

        mouth_path = (
            mouth_dir /
            f"frame_{frame_index:06d}.jpg"
        )


        if mouth_path.exists():

            stats["mouth_saved"] += 1

            rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "aligned_face":
                    str(
                        aligned_path
                    ),

                "mouth_crop":
                    str(
                        mouth_path
                    ),

                "status":
                    "OK",

                "reason":
                    "already_exists",

            })

            continue


        # ----------------------------------------------------
        # Read aligned face
        # ----------------------------------------------------

        image = cv2.imread(
            str(aligned_path)
        )


        if image is None:

            stats["errors"] += 1

            rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "aligned_face":
                    str(
                        aligned_path
                    ),

                "mouth_crop":
                    "",

                "status":
                    "SKIPPED",

                "reason":
                    "image_read_error",

            })

            continue


        # ----------------------------------------------------
        # MediaPipe
        # ----------------------------------------------------

        try:

            mouth, reason = (
                crop_mouth_from_aligned_face(
                    image
                )
            )

        except Exception as e:

            stats["errors"] += 1

            rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "aligned_face":
                    str(
                        aligned_path
                    ),

                "mouth_crop":
                    "",

                "status":
                    "SKIPPED",

                "reason":
                    f"exception:{type(e).__name__}",

            })

            continue


        # ----------------------------------------------------
        # Crop failed
        # ----------------------------------------------------

        if mouth is None:

            stats["skipped"] += 1

            if reason == "no_landmarks":

                stats["no_landmarks"] += 1

            elif reason in {
                "invalid_crop",
                "empty_crop",
            }:

                stats["invalid_crop"] += 1


            rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "aligned_face":
                    str(
                        aligned_path
                    ),

                "mouth_crop":
                    "",

                "status":
                    "SKIPPED",

                "reason":
                    reason,

            })

            continue


        # ----------------------------------------------------
        # Save mouth crop
        # ----------------------------------------------------

        success = cv2.imwrite(

            str(mouth_path),

            mouth,

            [
                cv2.IMWRITE_JPEG_QUALITY,
                95
            ]

        )


        if not success:

            stats["errors"] += 1

            rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "aligned_face":
                    str(
                        aligned_path
                    ),

                "mouth_crop":
                    "",

                "status":
                    "SKIPPED",

                "reason":
                    "save_error",

            })

            continue


        stats["mouth_saved"] += 1


        rows.append({

            "video_id":
                video_id,

            "frame_index":
                frame_index,

            "aligned_face":
                str(
                    aligned_path
                ),

            "mouth_crop":
                str(
                    mouth_path
                ),

            "status":
                "OK",

            "reason":
                "mediapipe_success",

        })


    # --------------------------------------------------------
    # Save metadata
    # --------------------------------------------------------

    result_df = pd.DataFrame(
        rows,
        columns=MOUTH_METADATA_COLUMNS
    )


    result_df = result_df.sort_values(
        "frame_index"
    ).reset_index(
        drop=True
    )


    result_df.to_csv(
        mouth_metadata_path,
        index=False,
        encoding="utf-8-sig"
    )


    print()
    print("=" * 70)
    print(
        f"VIDEO: {video_id}"
    )
    print("=" * 70)

    print(
        "Input frames :",
        stats["input_frames"]
    )

    print(
        "Mouth saved  :",
        stats["mouth_saved"]
    )

    print(
        "Skipped      :",
        stats["skipped"]
    )

    print(
        "No landmarks :",
        stats["no_landmarks"]
    )

    print(
        "Invalid crop :",
        stats["invalid_crop"]
    )

    print(
        "Errors       :",
        stats["errors"]
    )

    print(
        "Metadata     :",
        mouth_metadata_path
    )


    return stats


print("=" * 70)
print("Cell 10 — Video Processor Ready")
print("=" * 70)

Cell 10 — Video Processor Ready


In [34]:
# ============================================================
# PHASE 4 — CELL 11
# Batch Runner
# ============================================================


def discover_phase3_videos():

    videos = []

    for path in sorted(
        PHASE4_OUTPUT_ROOT.iterdir(),
        key=lambda p: p.name.lower()
    ):

        if not path.is_dir():
            continue

        aligned_dir = (
            path / "aligned_face"
        )

        metadata = (
            path /
            "alignment_metadata.csv"
        )

        if (
            aligned_dir.exists()
            and
            metadata.exists()
        ):

            videos.append(
                path
            )

    return videos


VIDEO_OUTPUT_DIRS = (
    discover_phase3_videos()
)


print()
print("=" * 70)
print("PHASE 4 — BATCH PROCESSING")
print("=" * 70)

print(
    "Videos found:",
    len(VIDEO_OUTPUT_DIRS)
)


if not VIDEO_OUTPUT_DIRS:

    raise RuntimeError(
        "No valid Phase 3 outputs found."
    )


ALL_STATS = []


for index, video_output_dir in enumerate(
    VIDEO_OUTPUT_DIRS,
    start=1
):

    print()
    print(
        f"[{index}/{len(VIDEO_OUTPUT_DIRS)}] "
        f"{video_output_dir.name}"
    )


    try:

        stats = process_video(
            video_output_dir
        )

        ALL_STATS.append(
            stats
        )


        print()
        print(
            f"COMPLETED: "
            f"{stats['video_id']}"
        )


    except Exception as e:

        print()
        print(
            f"FAILED: "
            f"{video_output_dir.name}"
        )

        print(
            "Error:",
            f"{type(e).__name__}: {e}"
        )

        traceback.print_exc()


    finally:

        gc.collect()


print()
print("=" * 70)
print("PHASE 4 — BATCH SUMMARY")
print("=" * 70)

for stats in ALL_STATS:

    print(
        f"{stats['video_id']:<15}"
        f"input={stats['input_frames']:<8}"
        f"mouth={stats['mouth_saved']:<8}"
        f"skipped={stats['skipped']:<8}"
        f"errors={stats['errors']}"
    )

print("=" * 70)


PHASE 4 — BATCH PROCESSING
Videos found: 3

[1/3] video001

PHASE 4 PROCESSING : video001
Aligned face : C:\LipReadingSSL\output\video001\aligned_face
Metadata     : C:\LipReadingSSL\output\video001\alignment_metadata.csv
Mouth crop   : C:\LipReadingSSL\output\video001\mouth_crop


C:\Users\User\AppData\Local\Temp\ipykernel_3120\1404112470.py:30: DtypeWarning: Columns (15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(



Metadata frames : 41058
Aligned frames  : 82450


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



VIDEO: video001
Input frames : 41058
Mouth saved  : 40746
Skipped      : 312
No landmarks : 312
Invalid crop : 0
Errors       : 0
Metadata     : C:\LipReadingSSL\output\video001\mouth_metadata.csv

COMPLETED: video001

[2/3] video002

PHASE 4 PROCESSING : video002
Aligned face : C:\LipReadingSSL\output\video002\aligned_face
Metadata     : C:\LipReadingSSL\output\video002\alignment_metadata.csv
Mouth crop   : C:\LipReadingSSL\output\video002\mouth_crop

Metadata frames : 34794
Aligned frames  : 73658


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



VIDEO: video002
Input frames : 34794
Mouth saved  : 34278
Skipped      : 516
No landmarks : 516
Invalid crop : 0
Errors       : 0
Metadata     : C:\LipReadingSSL\output\video002\mouth_metadata.csv

COMPLETED: video002

[3/3] video003

PHASE 4 PROCESSING : video003
Aligned face : C:\LipReadingSSL\output\video003\aligned_face
Metadata     : C:\LipReadingSSL\output\video003\alignment_metadata.csv
Mouth crop   : C:\LipReadingSSL\output\video003\mouth_crop

Metadata frames : 28276
Aligned frames  : 28276


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



VIDEO: video003
Input frames : 28276
Mouth saved  : 28110
Skipped      : 166
No landmarks : 166
Invalid crop : 0
Errors       : 0
Metadata     : C:\LipReadingSSL\output\video003\mouth_metadata.csv

COMPLETED: video003

PHASE 4 — BATCH SUMMARY
video001       input=41058   mouth=40746   skipped=312     errors=0
video002       input=34794   mouth=34278   skipped=516     errors=0
video003       input=28276   mouth=28110   skipped=166     errors=0


In [4]:
# ============================================================
# PHASE 4 — VALIDATION DEPENDENCIES
# ============================================================

from pathlib import Path
import pandas as pd


# ============================================================
# Root
# ============================================================

OUTPUT_ROOT = Path(
    r"C:\LipReadingSSL\output"
)


# ============================================================
# Phase 4 Paths
# ============================================================

def get_phase4_paths(
    video_output_dir
):

    video_output_dir = Path(
        video_output_dir
    ).resolve()

    return {

        "output":
            video_output_dir,

        "aligned_face":
            video_output_dir /
            "aligned_face",

        "alignment_metadata":
            video_output_dir /
            "alignment_metadata.csv",

        "mouth_crop":
            video_output_dir /
            "mouth_crop",

        "mouth_metadata":
            video_output_dir /
            "mouth_metadata.csv",
    }


# ============================================================
# Discover aligned frames
# ============================================================

def discover_aligned_frames(
    aligned_face_dir
):

    aligned_face_dir = Path(
        aligned_face_dir
    )

    frame_map = {}

    if not aligned_face_dir.exists():
        return frame_map

    for file_path in aligned_face_dir.iterdir():

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() != ".jpg":
            continue

        name = file_path.stem

        # รองรับ:
        # frame_000000
        if name.startswith("frame_"):

            try:

                frame_index = int(
                    name.replace(
                        "frame_",
                        "",
                        1
                    )
                )

                frame_map[
                    frame_index
                ] = file_path

            except ValueError:
                pass

    return frame_map


# ============================================================
# Load alignment metadata
# ============================================================

def load_alignment_metadata(
    metadata_path
):

    metadata_path = Path(
        metadata_path
    )

    if not metadata_path.exists():

        raise FileNotFoundError(
            f"Alignment metadata not found:\n"
            f"{metadata_path}"
        )

    df = pd.read_csv(
        metadata_path,
        low_memory=False
    )

    if "frame_index" not in df.columns:

        raise KeyError(
            "Column 'frame_index' "
            "not found in alignment metadata."
        )

    df["frame_index"] = pd.to_numeric(
        df["frame_index"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["frame_index"]
    )

    df["frame_index"] = (
        df["frame_index"]
        .astype(int)
    )

    return df


print("=" * 70)
print("PHASE 4 VALIDATION DEPENDENCIES READY")
print("=" * 70)

PHASE 4 VALIDATION DEPENDENCIES READY


In [5]:
# ============================================================
# PHASE 4 — CELL 12
# Final Validation
# ============================================================

from pathlib import Path
import pandas as pd


# ============================================================
# Configuration
# ============================================================

OUTPUT_ROOT = Path(
    r"C:\LipReadingSSL\output"
)

VIDEO_IDS = [
    "video001",
    "video002",
    "video003",
]

VIDEO_OUTPUT_DIRS = [
    OUTPUT_ROOT / video_id
    for video_id in VIDEO_IDS
]


# ============================================================
# Check output directories
# ============================================================

print("=" * 70)
print("PHASE 4 — VALIDATION INPUT")
print("=" * 70)

for video_output_dir in VIDEO_OUTPUT_DIRS:

    print(
        f"{video_output_dir.name}: "
        f"{video_output_dir.exists()}"
    )

    if not video_output_dir.exists():
        raise FileNotFoundError(
            f"Video output directory not found:\n"
            f"{video_output_dir}"
        )


# ============================================================
# Validation Function
# ============================================================

def validate_phase4_video(
    video_output_dir
):

    video_output_dir = Path(
        video_output_dir
    ).resolve()

    video_id = (
        video_output_dir.name
    )

    # --------------------------------------------------------
    # Get Phase 4 paths
    # --------------------------------------------------------

    paths = get_phase4_paths(
        video_output_dir
    )

    # --------------------------------------------------------
    # Discover aligned frames
    # --------------------------------------------------------

    aligned_map = (
        discover_aligned_frames(
            paths["aligned_face"]
        )
    )

    # --------------------------------------------------------
    # Load alignment metadata
    # --------------------------------------------------------

    metadata_df = (
        load_alignment_metadata(
            paths["alignment_metadata"]
        )
    )

    # --------------------------------------------------------
    # Load mouth metadata
    # --------------------------------------------------------

    mouth_df = pd.DataFrame()

    if paths["mouth_metadata"].exists():

        mouth_df = pd.read_csv(
            paths["mouth_metadata"],
            low_memory=False
        )

    # ========================================================
    # Frame sets
    # ========================================================

    aligned_frames = set(
        aligned_map.keys()
    )

    metadata_frames = set(
        pd.to_numeric(
            metadata_df["frame_index"],
            errors="coerce"
        )
        .dropna()
        .astype(int)
    )

    mouth_frames = set()

    if (
        not mouth_df.empty
        and
        "frame_index" in mouth_df.columns
        and
        "status" in mouth_df.columns
    ):

        mouth_frames = set(
            pd.to_numeric(
                mouth_df[
                    mouth_df["status"] == "OK"
                ]["frame_index"],
                errors="coerce"
            )
            .dropna()
            .astype(int)
        )

    # ========================================================
    # Expected frames
    # ========================================================

    # Frames ที่ metadata และ aligned face มีตรงกัน
    expected_frames = (
        metadata_frames
        & aligned_frames
    )

    # Frames ที่ควรมี mouth แต่ไม่มี mouth
    skipped_frames = (
        expected_frames
        - mouth_frames
    )

    # ========================================================
    # Invalid relationships
    # ========================================================

    aligned_without_metadata = (
        aligned_frames
        - metadata_frames
    )

    mouth_without_aligned = (
        mouth_frames
        - aligned_frames
    )

    mouth_without_metadata = (
        mouth_frames
        - metadata_frames
    )

    # ========================================================
    # Duplicate mouth metadata
    # ========================================================

    metadata_duplicates = 0

    if (
        not mouth_df.empty
        and
        "frame_index" in mouth_df.columns
    ):

        metadata_duplicates = int(
            mouth_df[
                "frame_index"
            ]
            .duplicated()
            .sum()
        )

    # ========================================================
    # Errors
    # ========================================================

    errors = []

    if aligned_without_metadata:

        errors.append(
            "aligned_without_metadata"
        )

    if mouth_without_aligned:

        errors.append(
            "mouth_without_aligned"
        )

    if mouth_without_metadata:

        errors.append(
            "mouth_without_metadata"
        )

    if metadata_duplicates:

        errors.append(
            "metadata_duplicates"
        )

    # ========================================================
    # Validate SKIPPED frames
    # ========================================================

    invalid_skipped = set()

    if (
        not mouth_df.empty
        and
        "status" in mouth_df.columns
        and
        "frame_index" in mouth_df.columns
    ):

        skipped_df = mouth_df[
            mouth_df["status"] == "SKIPPED"
        ]

        skipped_frames_recorded = set(
            pd.to_numeric(
                skipped_df["frame_index"],
                errors="coerce"
            )
            .dropna()
            .astype(int)
        )

        invalid_skipped = (
            skipped_frames_recorded
            - expected_frames
        )

    if invalid_skipped:

        errors.append(
            "invalid_skipped_frames"
        )

    # ========================================================
    # Final status
    # ========================================================

    final_ok = (
        len(errors) == 0
        and
        len(metadata_frames) > 0
        and
        len(aligned_frames) == len(metadata_frames)
    )

    # ========================================================
    # Result
    # ========================================================

    result = {

        "video_id":
            video_id,

        "aligned_frames":
            len(aligned_frames),

        "metadata_frames":
            len(metadata_frames),

        "mouth_ok":
            len(mouth_frames),

        "expected_frames":
            len(expected_frames),

        "skipped_expected":
            len(skipped_frames),

        "aligned_no_metadata":
            len(
                aligned_without_metadata
            ),

        "mouth_no_aligned":
            len(
                mouth_without_aligned
            ),

        "mouth_no_metadata":
            len(
                mouth_without_metadata
            ),

        "metadata_duplicates":
            metadata_duplicates,

        "errors":
            ";".join(errors),

        "final_ok":
            final_ok,
    }

    return result


# ============================================================
# Run Validation
# ============================================================

FINAL_ROWS = []

for video_output_dir in VIDEO_OUTPUT_DIRS:

    print()
    print(
        f"Validating: "
        f"{video_output_dir.name}"
    )

    result = validate_phase4_video(
        video_output_dir
    )

    FINAL_ROWS.append(
        result
    )


# ============================================================
# Create DataFrame
# ============================================================

FINAL_VALIDATION_DF = pd.DataFrame(
    FINAL_ROWS
)


# ============================================================
# Print Result
# ============================================================

print()
print("=" * 70)
print("PHASE 4 — FINAL VALIDATION")
print("=" * 70)

if not FINAL_VALIDATION_DF.empty:

    print(
        FINAL_VALIDATION_DF.to_string(
            index=False
        )
    )

print()
print("=" * 70)


# ============================================================
# Final Decision
# ============================================================

if (
    not FINAL_VALIDATION_DF.empty
    and
    FINAL_VALIDATION_DF["final_ok"].all()
):

    print(
        "✅ PHASE 4 VALIDATION PASSED"
    )

else:

    print(
        "⚠️ PHASE 4 VALIDATION REQUIRES ATTENTION"
    )

print("=" * 70)

PHASE 4 — VALIDATION INPUT
video001: True
video002: True
video003: True

Validating: video001

Validating: video002

Validating: video003

PHASE 4 — FINAL VALIDATION
video_id  aligned_frames  metadata_frames  mouth_ok  expected_frames  skipped_expected  aligned_no_metadata  mouth_no_aligned  mouth_no_metadata  metadata_duplicates errors  final_ok
video001           41058            41058     40746            41058               312                    0                 0                  0                    0             True
video002           34794            34794     34278            34794               516                    0                 0                  0                    0             True
video003           28276            28276     28110            28276               166                    0                 0                  0                    0             True

✅ PHASE 4 VALIDATION PASSED
